<a href="https://colab.research.google.com/github/DuvvuLakshmiPrasanna/JSON-R-sum--Extractor-2B/blob/main/Day2_ResumeExtractor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q google-genai pydantic

import os, getpass

if 'GEMINI_API_KEY' not in os.environ:
    os.environ['GEMINI_API_KEY'] = getpass.getpass('Gemini API key: ')

Gemini API key: ··········


In [2]:
from pydantic import BaseModel
from typing import List, Optional

class Education(BaseModel):
    degree: str
    institution: str
    year: int

class Resume(BaseModel):
    name: str
    email: str
    phone: Optional[str] = None
    education: List[Education]
    skills: List[str]
    projects: List[str] = []
    experience_years: float

In [3]:
from google import genai
from pydantic import ValidationError

client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])

def extract_resume(raw_text: str, max_retries: int = 1):
    for attempt in range(max_retries + 1):
        try:
            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=f"""
                Extract a Resume JSON from this text.
                Return ONLY JSON.

                {raw_text}
                """,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

        except ValidationError as e:

            if attempt == max_retries:
                raise

            fix_prompt = (
                f"Fix this JSON to match schema. Errors: {e}. "
                f"Original: {resp.text}"
            )

            resp = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=fix_prompt,
                config={
                    "response_mime_type": "application/json",
                    "response_schema": Resume.model_json_schema(),
                },
            )

            return Resume.model_validate_json(resp.text)

In [4]:
from google.colab import files
uploaded = files.upload()

Saving sample_resumes.txt to sample_resumes.txt


In [5]:
with open("sample_resumes.txt") as f:
    resumes = [
        r.strip()
        for r in f.read().split("---")
        if r.strip()
    ]

print("Loaded", len(resumes), "resumes")

Loaded 5 resumes


In [9]:
results = []

for i, r in enumerate(resumes):
    try:
        parsed = extract_resume(r)

        results.append(parsed)

        print(
            f"Resume {i+1}: {parsed.name} - "
            f"{len(parsed.skills)} skills - "
            f"{parsed.experience_years} years exp"
        )

    except Exception as e:
        print(
            f"Resume {i+1}: FAILED - "
            f"{type(e).__name__}: {e}"
        )

Resume 1: Ravi Kumar - 6 skills - 1.0 years exp
Resume 2: Sneha Reddy - 6 skills - 0.5 years exp
Resume 3: Arun Pillai - 9 skills - 1.0 years exp
Resume 4: Priya Nair - 7 skills - 1.0 years exp
Resume 5: Karthik Subramaniam - 10 skills - 1.5 years exp


In [10]:
for i, resume in enumerate(results):
    print(f"\n===== Resume {i+1} =====")
    print(resume.model_dump_json(indent=2))


===== Resume 1 =====
{
  "name": "Ravi Kumar",
  "email": "ravi.kumar@email.com",
  "phone": "+91 98765 43210",
  "education": [
    {
      "degree": "B.Tech in Computer Science",
      "institution": "IIT Hyderabad",
      "year": 2023
    }
  ],
  "skills": [
    "Python",
    "REST APIs",
    "SQL",
    "Git",
    "Linux",
    "Docker"
  ],
  "projects": [
    "Student Portal: Built a Django-based student attendance management system for 500+ users",
    "Weather Dashboard: React + OpenWeatherMap API integration with live charts"
  ],
  "experience_years": 1.0
}

===== Resume 2 =====
{
  "name": "Sneha Reddy",
  "email": "sneha.reddy@gmail.com",
  "phone": null,
  "education": [
    {
      "degree": "B.Sc. in Information Technology",
      "institution": "Osmania University",
      "year": 2024
    }
  ],
  "skills": [
    "HTML",
    "CSS",
    "JavaScript",
    "React",
    "Figma",
    "Canva"
  ],
  "projects": [
    "NGO Website",
    "Portfolio Builder"
  ],
  "experience_y